# Лабораторная работа 3. 

**LLM:** локальная модель через [LM Studio](https://lmstudio.ai/) (OpenAI-compatible API, `http://localhost:1234/v1`).

## 0. Установка зависимостей и LM Studio

In [ ]:
!pip install -q requests pandas matplotlib

In [ ]:
import json
import os
import re
import time
from dataclasses import dataclass, field
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import pandas as pd
import requests


LM_STUDIO_BASE = os.environ.get("LM_STUDIO_BASE", "http://localhost:1234/v1")
MODEL_NAME = os.environ.get("LM_STUDIO_MODEL", "openai/gpt-oss-20b")

print(f"LM Studio: {LM_STUDIO_BASE}, model={MODEL_NAME}")

SYSTEM_PROMPT = """You are a scientific research assistant.
Your task is to prepare structured, factual literature reviews based on provided sources.
Always cite only information supported by the given context and sources.
Respond in the same language as the user prompt (Russian if the prompt is in Russian).
Use clear section headings and avoid hallucinating references not present in the sources."""

## 1. Вызов LLM через LM Studio

In [ ]:
def llm_call(prompt: str, temperature: float = 0.3) -> str:
    response = requests.post(
        url=f"{LM_STUDIO_BASE}/chat/completions",
        headers={"Content-Type": "application/json"},
        json={
            "model": MODEL_NAME,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt[:12000]},
            ],
            "temperature": temperature,
            "max_tokens": 3000,
        },
        timeout=600,
    )
    if response.status_code != 200:
        raise Exception(f"LM Studio error {response.status_code}: {response.text}")
    return response.json()["choices"][0]["message"]["content"]

## 2. Инструменты: Wikipedia, OpenAlex, обработка аннотаций

In [ ]:
WIKI_HEADERS = {"User-Agent": "Lab3AgentAI/1.0 (scientific homework)"}


def search_wikipedia(query: str) -> str:
    title = query.replace(" ", "_")
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}"
    response = requests.get(url, headers=WIKI_HEADERS, timeout=30)
    if response.status_code == 200:
        return response.json().get("extract", "")

    search_url = "https://en.wikipedia.org/w/api.php"
    search_resp = requests.get(
        search_url,
        headers=WIKI_HEADERS,
        params={
            "action": "query",
            "list": "search",
            "srsearch": query,
            "format": "json",
            "srlimit": 1,
        },
        timeout=30,
    )
    search_resp.raise_for_status()
    results = search_resp.json().get("query", {}).get("search", [])
    if not results:
        return ""

    best_title = results[0]["title"].replace(" ", "_")
    summary_resp = requests.get(
        f"https://en.wikipedia.org/api/rest_v1/page/summary/{best_title}",
        headers=WIKI_HEADERS,
        timeout=30,
    )
    if summary_resp.status_code != 200:
        return ""
    return summary_resp.json().get("extract", "")


def search_openalex(query: str, per_page: int = 5, page: int = 1) -> Tuple[List[dict], int]:
    url = "https://api.openalex.org/works"
    params = {
        "search": query,
        "per-page": per_page,
        "page": page,
        "select": "id,display_name,publication_year,abstract_inverted_index,authorships",
    }
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()
    return data.get("results", []), data.get("meta", {}).get("count", 0)


def invert_abstract(inv_idx: dict) -> str:
    if not inv_idx:
        return ""
    words = []
    for token, positions in inv_idx.items():
        for pos in positions:
            words.append((pos, token))
    words.sort(key=lambda x: x[0])
    return " ".join(token for _, token in words)

## 3. Состояние агента и трассировка

In [ ]:
@dataclass
class AgentState:
    topic: str
    objective: str = "Prepare structured scientific research"
    step_id: int = 0
    history: List[Dict[str, Any]] = field(default_factory=list)
    sources: List[Dict] = field(default_factory=list)
    notes: List[Dict] = field(default_factory=list)
    final_answer: str = ""
    status: str = "running"
    stop_reason: str = ""


def log_step(state: AgentState, action: str, payload: dict, result_preview: str):
    state.history.append(
        {
            "step_id": state.step_id,
            "action": action,
            "payload": payload,
            "result_preview": result_preview[:500],
        }
    )
    state.step_id += 1


def save_trace(state: AgentState, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "topic": state.topic,
                "history": state.history,
                "n_sources": len(state.sources),
                "n_notes": len(state.notes),
                "status": state.status,
                "stop_reason": state.stop_reason,
                "final_answer_preview": state.final_answer[:1000],
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

## 4. Baseline-режим

In [ ]:
def run_baseline(topic: str, llm_call_func) -> str:
    wiki_context = search_wikipedia(topic)
    prompt = f"""Сформируй краткий научно-аналитический обзор по теме: {topic}.

Используй следующий справочный контекст из Wikipedia:
{wiki_context or "(контекст не найден)"}

Обязательная структура ответа:
1) определение темы
2) основные подходы
3) 3-5 ключевых работ (если данных недостаточно — укажи это явно)
4) возможные применения
5) ограничения
6) список использованных источников

Пиши структурированно, без выдуманных ссылок."""
    return llm_call_func(prompt)

## 5. Agent-режим

In [ ]:
def run_agent(topic: str, llm_call_func, per_page: int = 5, max_steps: int = 6) -> AgentState:
    state = AgentState(topic=topic)

    wiki = search_wikipedia(topic)
    log_step(state, "wikipedia", {"topic": topic}, wiki[:200])

    if state.step_id >= max_steps:
        state.final_answer = "Недостаточно шагов для выполнения."
        state.status = "failed"
        state.stop_reason = "max_steps_exceeded"
        return state

    all_notes = []
    all_papers = []
    page = 1
    total_available = 0
    max_search_iter = max(1, (max_steps - 2) // 2)

    for _ in range(max_search_iter):
        if state.step_id + 2 > max_steps:
            break

        papers, total = search_openalex(topic, per_page=per_page, page=page)
        if not papers:
            break

        total_available = total
        all_papers.extend(papers)
        log_step(
            state,
            f"openalex_search_page_{page}",
            {"query": topic, "per_page": per_page, "page": page},
            f"found {len(papers)} papers (total {total})",
        )

        notes_page = []
        for p in papers:
            abstract = invert_abstract(p.get("abstract_inverted_index", {}))
            if abstract:
                notes_page.append(
                    {
                        "title": p.get("display_name", "Untitled"),
                        "year": p.get("publication_year", ""),
                        "abstract": abstract[:1200],
                    }
                )

        if notes_page:
            all_notes.extend(notes_page)
            log_step(
                state,
                f"extract_abstracts_page_{page}",
                {"n": len(notes_page)},
                f"total notes: {len(all_notes)}",
            )
        else:
            log_step(
                state,
                f"extract_abstracts_page_{page}",
                {"n": 0},
                "no abstracts found",
            )

        if len(all_notes) >= 10 or page * per_page >= total_available:
            break
        page += 1

    state.sources = all_papers
    state.notes = all_notes

    if not state.notes:
        state.final_answer = "Не найдено ни одной аннотации для формирования ответа."
        state.status = "failed"
        state.stop_reason = "no_sources"
        return state

    if state.step_id >= max_steps:
        state.final_answer = "Достигнут лимит шагов до генерации."
        state.status = "failed"
        state.stop_reason = "max_steps_exceeded"
        return state

    prompt = f"""Подготовь научно-аналитический обзор по теме: {topic}

Общий контекст (Wikipedia):
{wiki or "(контекст не найден)"}

Источники из OpenAlex (используй только их):
{json.dumps(state.notes, ensure_ascii=False)}

Обязательная структура:
- определение темы
- основные подходы
- 3-5 ключевых работ с названием и годом
- возможные применения
- ограничения
- список использованных источников

Не добавляй источники, которых нет в списке выше."""

    state.final_answer = llm_call_func(prompt)
    state.status = "finished"
    state.stop_reason = "final_answer_generated"
    log_step(
        state,
        "generate_final",
        {"total_sources": len(state.notes)},
        state.final_answer[:300],
    )
    return state

## 6. Evaluator

In [ ]:
EVAL_PROMPT = """Оцени ответ по шкале от 0 до 5 по критериям:
1. correctness
2. groundedness
3. completeness
4. coverage_of_required_fields
5. source_consistency

Верни только JSON:
{
  "correctness": int,
  "groundedness": int,
  "completeness": int,
  "coverage_of_required_fields": int,
  "source_consistency": int,
  "comment": "..."
}"""


def _extract_json(raw: str) -> dict:
    raw = raw.strip()
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", raw, re.DOTALL)
    if fenced:
        raw = fenced.group(1)
    start, end = raw.find("{"), raw.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"JSON not found in evaluator response: {raw[:300]}")
    return json.loads(raw[start : end + 1])


def evaluate_answer(answer: str, notes: list, llm_call_func) -> dict:
    prompt = (
        EVAL_PROMPT
        + "\n\nОтвет:\n"
        + answer
        + "\n\nИсточники:\n"
        + json.dumps(notes, ensure_ascii=False)
    )
    raw = llm_call_func(prompt, temperature=0.0)
    return _extract_json(raw)


def run_agent_with_evaluator(topic: str, llm_call_func) -> Tuple[AgentState, dict]:
    state = run_agent(topic, llm_call_func)
    scores = evaluate_answer(state.final_answer, state.notes, llm_call_func)

    improve_prompt = f"""Улучши научно-аналитический обзор по теме "{topic}".

Оценки evaluator:
{json.dumps(scores, ensure_ascii=False)}

Исходный ответ:
{state.final_answer}

Источники:
{json.dumps(state.notes, ensure_ascii=False)}

Сохрани обязательную структуру (определение, подходы, 3-5 работ, применения, ограничения, источники).
Исправь слабые места, отмеченные в comment, и не добавляй выдуманные ссылки."""

    improved = llm_call_func(improve_prompt)
    state.final_answer = improved
    state.stop_reason = "regenerated_after_eval"
    log_step(state, "regenerate_after_eval", {"scores": scores}, improved[:300])
    return state, scores

## 7. Цикл экспериментов

In [ ]:
def run_experiment(topics, mode_name, runner_func, llm_call_func):
    records = []
    for topic in topics:
        print(f"  Running {mode_name} on '{topic}'...")
        start = time.time()
        result = runner_func(topic, llm_call_func)
        latency = time.time() - start

        if isinstance(result, tuple):
            state, eval_dict = result
            answer = state.final_answer
            notes = state.notes
            n_steps = len(state.history)
            stop_reason = state.stop_reason
        elif hasattr(result, "final_answer"):
            state = result
            answer = state.final_answer
            notes = state.notes
            n_steps = len(state.history)
            stop_reason = state.stop_reason
            eval_dict = evaluate_answer(answer, notes, llm_call_func)
        else:
            answer = result
            notes = []
            n_steps = 1
            stop_reason = "baseline_single_step"
            eval_dict = evaluate_answer(answer, notes, llm_call_func)

        rubric = sum(
            eval_dict[k]
            for k in [
                "correctness",
                "groundedness",
                "completeness",
                "coverage_of_required_fields",
                "source_consistency",
            ]
        ) / 5.0

        records.append(
            {
                "topic": topic,
                "mode": mode_name,
                "correctness": eval_dict["correctness"],
                "groundedness": eval_dict["groundedness"],
                "completeness": eval_dict["completeness"],
                "coverage": eval_dict["coverage_of_required_fields"],
                "source_consistency": eval_dict["source_consistency"],
                "rubric_score": rubric,
                "n_steps": n_steps,
                "latency": latency,
                "stop_reason": stop_reason,
            }
        )
    return pd.DataFrame(records)

## 8. Запуск основных экспериментов


In [ ]:
TOPICS = [
    "Agentic AI for customer support",
    "Graph RAG for enterprise knowledge systems",
    "LLM evaluation and process-aware metrics",
    "Tool-using language models in scientific search",
    "Retrieval-augmented generation in medicine",
    "Planning and reflection in LLM agents",
    "Human-in-the-loop AI systems",
    "Knowledge graphs for procedural reasoning",
]

QUICK_TEST = False
EXPERIMENT_TOPICS = TOPICS[:2] if QUICK_TEST else TOPICS

print("Start baseline experiments...")
df_baseline = run_experiment(EXPERIMENT_TOPICS, "baseline", run_baseline, llm_call)

print("Start agent experiments...")
df_agent = run_experiment(EXPERIMENT_TOPICS, "agent", run_agent, llm_call)

print("Start eval experiments...")
df_agent_eval = run_experiment(EXPERIMENT_TOPICS, "eval", run_agent_with_evaluator, llm_call)

df_all = pd.concat([df_baseline, df_agent, df_agent_eval], ignore_index=True)

OUTPUT_DIR = "/content" if os.path.exists("/content") else "."
results_path = os.path.join(OUTPUT_DIR, "results.csv")
df_all.to_csv(results_path, index=False)
print(f"Saved: {results_path}")
df_all

## 9. Дополнительные эксперименты (число источников и max_steps)

In [ ]:
def run_agent_topk(topic: str, llm_call_func, topk: int = 5):
    return run_agent(topic, llm_call_func, per_page=topk, max_steps=8)


def run_agent_maxsteps(topic: str, llm_call_func, max_steps: int = 6):
    return run_agent(topic, llm_call_func, per_page=5, max_steps=max_steps)


extra_records = []
probe_topics = EXPERIMENT_TOPICS[:2]

for topk in [3, 5, 8]:
    runner = lambda topic, fn, k=topk: run_agent_topk(topic, fn, topk=k)
    df_k = run_experiment(probe_topics, f"agent_top{topk}", runner, llm_call)
    extra_records.append(df_k)

for steps in [4, 6, 8]:
    runner = lambda topic, fn, s=steps: run_agent_maxsteps(topic, fn, max_steps=s)
    df_s = run_experiment(probe_topics, f"agent_steps{steps}", runner, llm_call)
    extra_records.append(df_s)

df_extra = pd.concat(extra_records, ignore_index=True)
df_extra.to_csv(os.path.join(OUTPUT_DIR, "results_extra.csv"), index=False)
df_extra

## 10. Агрегация и визуализация

In [ ]:
summary = df_all.groupby("mode")[
    ["correctness", "groundedness", "completeness", "coverage", "rubric_score", "n_steps", "latency"]
].mean()
print(summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

summary[["correctness", "groundedness", "completeness", "rubric_score"]].plot(
    kind="bar", ax=axes[0], rot=0
)
axes[0].set_title("Сравнение качества результатов")
axes[0].set_ylabel("Средний балл")

summary[["n_steps", "latency"]].plot(kind="bar", ax=axes[1], rot=0)
axes[1].set_title("Сравнение процесса выполнения")
axes[1].set_ylabel("Значение")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_plots.png"), dpi=150)
plt.show()

## 11. Сохранение trace для разбора ошибок

In [ ]:
trace_dir = os.path.join(OUTPUT_DIR, "traces")
os.makedirs(trace_dir, exist_ok=True)

for topic in EXPERIMENT_TOPICS[:3]:
    state = run_agent(topic, llm_call)
    safe_name = re.sub(r"[^a-zA-Z0-9_]+", "_", topic)[:60]
    save_trace(state, os.path.join(trace_dir, f"trace_{safe_name}.json"))

print(f"Traces saved to {trace_dir}")